In [ ]:
!pip install transformers datasets evaluate accelerate torch scikit-learn pandas openpyxl

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, pipeline
from datasets import Dataset, DatasetDict
from google.colab import drive
import numpy as np
import evaluate
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
drive.mount('/content/drive')

path = '/content/drive/MyDrive/SentimenAnalisis/dataset_augmented_v2.xlsx'

df = pd.read_excel(path)
df.head()

In [ ]:
df = df.dropna()

df['sentiment'].value_counts().sum()

In [ ]:
df = df[['Komentar', 'sentiment']]

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

model_checkpoint = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
  return tokenizer(examples['Komentar'], padding= 'max_length', truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

tokenized_datasets = tokenized_datasets.remove_columns(['Komentar', '__index_level_0__'])
tokenized_datasets = tokenized_datasets.rename_column('sentiment', 'labels')
tokenized_datasets.set_format('torch')

print(tokenized_datasets)

In [ ]:
id2label = {0: "Netral", 1: "Positif", 2: "Negatif"}
label2id = {"Netral": 0, "Positif": 1, "Negatif": 2}

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = metric_acc.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")["f1"]

    return {"accuracy": accuracy, "f1_macro": f1}

training_args = TrainingArguments(
    output_dir="./hasil_indobert_sentimen",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=6,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

print("Memulai proses Fine-Tuning IndoBERT")
trainer.train()

In [ ]:
output_prediksi = trainer.predict(tokenized_datasets["test"])

tebakan_indobert = np.argmax(output_prediksi.predictions, axis=-1)
label_asli = output_prediksi.label_ids

print("HASIL EVALUASI INDOBERT")
print(classification_report(label_asli, tebakan_indobert, target_names=["Netral (0)", "Positif (1)", "Negatif (2)"]))

In [ ]:
cm_bert = confusion_matrix(label_asli, tebakan_indobert)
label_names = ['Netral (0)', 'Positif (1)', 'Negatif (2)']

plt.figure(figsize=(7, 5))
sns.heatmap(cm_bert, annot=True, fmt='d', cmap='Greens',
            xticklabels=label_names,
            yticklabels=label_names)
plt.title('Confusion Matrix - IndoBERT', fontsize=14)
plt.xlabel('Tebakan Model (Predicted Label)', fontsize=12)
plt.ylabel('Data Asli (True Label)', fontsize=12)
plt.show()

In [ ]:
nlp_pipeline = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

def tebak_sentimen_bert(kalimat):
    hasil = nlp_pipeline(kalimat)[0]
    label = hasil['label']
    score = hasil['score']

    print(f"Komentar: '{kalimat}'")
    print(f"Tebakan IndoBERT: {label} (Tingkat Keyakinan: {score * 100:.2f}%)")
    print("-" * 40)

tebak_sentimen_bert("Wah mantap banget hpnya, baru 5 menit langsung panas")
tebak_sentimen_bert("kamera bagus, tapi baterainya jelek banget anjir")
tebak_sentimen_bert("coba main mobile legend pakai j2 prime bang")
tebak_sentimen_bert('Performanya stabil, multitasking aman')
tebak_sentimen_bert("Keren banget kameranya, muka saya jadi kayak CCTV 2007")
tebak_sentimen_bert('Terbaik lah hp ini')
tebak_sentimen_bert("Najis punya hp kek gini")
tebak_sentimen_bert("Update terbaru ini sangat membantu, sekarang aplikasinya malah gak bisa dibuka.")
tebak_sentimen_bert("Gaming phone terbaik, buka game aja patah-patah.")
tebak_sentimen_bert("Speaker-nya ternyata nendang juga.")
tebak_sentimen_bert("Battery awet banget, dipakai seharian masih sisa.")
tebak_sentimen_bert('keren banget')
tebak_sentimen_bert('suka si iphone yang kuning ini')

In [ ]:
model.save_pretrained("./model_indobert_final")
tokenizer.save_pretrained("./model_indobert_final")

In [ ]:
!zip -r model_indobert_final.zip ./model_indobert_final